In [ ]:
import urllib.request
import zipfile
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from tempfile import mkdtemp

import mlflow
import mlflow.sklearn


# URL RAW del archivo
url = "https://raw.githubusercontent.com/Mafegz0/Data/main/df_morosidad.csv.zip"

# Ruta temporal en Databricks
zip_path = "/tmp/df_morosidad.csv.zip"

# Descargar el ZIP
urllib.request.urlretrieve(url, zip_path)

# Descomprimir el ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/tmp")

# Cargar el CSV descomprimido
df = pd.read_csv("/tmp/df_morosidad.csv")
TARGET = "y_categorica"
DROP = ["Llave2", "Nombre_linea", "IDBANNER"]

y = df[TARGET].copy()
X = df.drop([TARGET] + [c for c in DROP if c in df.columns], axis=1).copy()

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)


prep = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ],
    remainder="drop"
)

xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    n_estimators=244,
    max_depth=5,
    learning_rate=0.0490198317,
    subsample=0.71825347433,
    colsample_bytree=0.63079196393,
    gamma=0.14487572645,
    min_child_weight=2
)

pipe = Pipeline([
    ("prep", prep),
    ("clf", xgb)
])
pipe.set_params(memory=mkdtemp())


experiment = mlflow.set_experiment("/modelo-xgboost-morosidad")

with mlflow.start_run(experiment_id=experiment.experiment_id):

    # ENTRENAR
    pipe.fit(X_train, y_train_enc)
    best_model = pipe

    # PROBABILIDADES
    proba_test = best_model.predict_proba(X_test)
    clases = list(le.classes_)
    y_test_bin = label_binarize(y_test_enc, classes=range(len(clases)))

    test_auc = roc_auc_score(
        y_test_bin, proba_test,
        multi_class="ovr", average="macro"
    )

    y_pred_enc = best_model.predict(X_test)
    y_pred = le.inverse_transform(y_pred_enc)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)


    mlflow.log_param("n_estimators",      xgb.n_estimators)
    mlflow.log_param("max_depth",         xgb.max_depth)
    mlflow.log_param("learning_rate",     xgb.learning_rate)
    mlflow.log_param("subsample",         xgb.subsample)
    mlflow.log_param("colsample_bytree",  xgb.colsample_bytree)
    mlflow.log_param("gamma",             xgb.gamma)
    mlflow.log_param("min_child_weight",  xgb.min_child_weight)


    mlflow.log_metric("test_auc_ovr_macro", float(test_auc))
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("precision_macro", float(prec))
    mlflow.log_metric("recall_macro", float(rec))


    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="modelo_xgb_morosidad"
    )

    print("\n=== RESULTADOS XGBOOST FINAL ===")
    print(f"AUC (OVR): {test_auc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision macro: {prec:.4f}")
    print(f"Recall macro: {rec:.4f}")
